# Arabic Google Play Reviews – Sentiment Analysis
## Saudi Play Store | Binary Classification | Two Models

**Research Question:** Can machine learning models accurately classify Google Play user reviews into positive and negative sentiments based on the text of the reviews?

**Hypothesis:** If we apply appropriate text preprocessing techniques and use machine learning algorithms such as Logistic Regression or Naive Bayes, then the model will be able to classify Google Play reviews into positive and negative sentiments with high accuracy.

---
### Apps Covered
| App | Sector |
|-----|--------|
| Absher | Government |
| STC Pay | Finance |
| HungerStation | Food Delivery |
| WhatsApp | Social Media |
| MySTC | Telecom |
| Snapchat | Social Media |

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
%matplotlib inline

Path('../outputs').mkdir(exist_ok=True)
Path('../models').mkdir(exist_ok=True)
print('Setup complete.')

## 1. Data Collection (Saudi Play Store, Arabic, Per-Star Sampling)

In [ ]:
# Option A: Scrape fresh data
# from data_collection import scrape_all_apps, save_dataset
# df_raw = scrape_all_apps(max_per_star=200)
# save_dataset(df_raw, '../data/reviews.csv')

# Option B: Load existing CSV
df_raw = pd.read_csv('../data/reviews.csv', encoding='utf-8-sig')
print(f'Loaded {len(df_raw):,} reviews')
df_raw.head(3)

## 2. Data Understanding

In [ ]:
print('Shape:', df_raw.shape)
print('\nDtype summary:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())

In [ ]:
# Star rating distribution per app
if 'appName' in df_raw.columns:
    pivot = df_raw.groupby(['appName', 'score']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', stacked=True, figsize=(12, 5),
               color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
    plt.title('Star Rating Distribution per App')
    plt.xlabel('App'); plt.ylabel('Count'); plt.legend(title='Stars')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 3. Preprocessing
- Arabic text cleaning (diacritics, tatweel, normalisation)
- Binary labeling: stars 1–2 = Negative, stars 4–5 = Positive, star 3 = dropped
- Stratified train/val/test split

In [ ]:
from preprocessing import extract_features, split_data, dataset_summary

df = extract_features(df_raw)
print(dataset_summary(df).to_string())

In [ ]:
# Show cleaning example
sample = df_raw['content'].dropna().sample(1).iloc[0]
from preprocessing import clean_arabic_text
print('Original:'), print(sample)
print('\nCleaned :'), print(clean_arabic_text(sample))

In [ ]:
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

In [ ]:
# Label balance after split
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    pos = (split['binary_label'] == 1).mean() * 100
    print(f'{name:5}: {pos:.1f}% positive, {100-pos:.1f}% negative')

## 4. Model 1 – TF-IDF + Logistic Regression
(Primary model per project hypothesis)

In [ ]:
from model_logistic_regression import train as train_lr, evaluate as eval_lr, plot_top_features

lr_model = train_lr(train_df, val_df, model_name='lr', save_dir='../models')
lr_metrics = eval_lr(lr_model, test_df, model_name='lr', plot_dir='../outputs')

In [ ]:
# Feature importance – what Arabic words drive sentiment?
pos_feat, neg_feat = plot_top_features(lr_model, n=15, plot_dir='../outputs')
print('Top Positive features:')
print(pos_feat.to_string(index=False))
print('\nTop Negative features:')
print(neg_feat.to_string(index=False))

## 4b. Naive Bayes (hypothesis comparison)

In [ ]:
nb_model = train_lr(train_df, val_df, model_name='nb', save_dir='../models')
nb_metrics = eval_lr(nb_model, test_df, model_name='nb', plot_dir='../outputs')

## 5. Model 2 – BiLSTM with Attention
(Bidirectional LSTM handles Arabic right-to-left morphology)

In [ ]:
from model_bilstm import train as train_lstm, evaluate as eval_lstm

lstm_model, lstm_tok = train_lstm(
    train_df, val_df,
    epochs=15,
    batch_size=64,
    save_dir='../models',
)
lstm_metrics = eval_lstm(lstm_model, test_df, lstm_tok, plot_dir='../outputs')

## 6. Model Comparison & Research Question Answer

In [ ]:
metrics_list  = ['accuracy', 'precision', 'recall', 'f1_macro', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'Macro F1', 'ROC-AUC']

comp = pd.DataFrame({
    'Metric':     metric_labels,
    'LR':         [lr_metrics[m]   for m in metrics_list],
    'NaiveBayes': [nb_metrics[m]   for m in metrics_list],
    'BiLSTM':     [lstm_metrics[m] for m in metrics_list],
})
print(comp.to_string(index=False))

In [ ]:
x = np.arange(len(metric_labels)); w = 0.25
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - w,  [lr_metrics[m]   for m in metrics_list], w, label='LR',         color='#3498db', alpha=0.85)
ax.bar(x,      [nb_metrics[m]   for m in metrics_list], w, label='Naive Bayes', color='#e67e22', alpha=0.85)
ax.bar(x + w,  [lstm_metrics[m] for m in metrics_list], w, label='BiLSTM',      color='#9b59b6', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Model Comparison – Arabic Google Play Sentiment (Saudi Store)')
ax.legend()
plt.tight_layout(); plt.show()

## 7. Live Inference Demo (Arabic)

In [ ]:
from preprocessing import clean_arabic_text
from model_logistic_regression import predict as lr_predict
from model_bilstm import predict as lstm_predict

test_reviews = [
    'التطبيق رائع جداً وسريع ويعمل بشكل ممتاز',       # great app
    'يتوقف عن العمل دائماً ولا يعمل بشكل صحيح',      # always crashes  
    'شكراً على هذا التطبيق المميز',                    # thanks great app
    'أسوأ تطبيق استخدمته في حياتي لا أنصح به',        # worst app ever
]

print(f'{"Review":<50} {"LR":>12} {"BiLSTM":>10}')
print('-' * 75)
for rev in test_reviews:
    cleaned = clean_arabic_text(rev)
    lr_pred   = lr_predict(lr_model,   [cleaned])[0]
    lstm_pred = lstm_predict(lstm_model, [cleaned], lstm_tok)[0]
    print(f'{rev[:48]:<50} {lr_pred:>12} {lstm_pred:>10}')

## 8. Known Limitations (Spec §5)

| Limitation | Impact | Mitigation Used |
|-----------|--------|----------------|
| Rating-Sentiment Inconsistency | Labels based on score, not actual text sentiment | Acknowledged; future: manual relabeling |
| Sarcasm (e.g., 'شكراً على إتلاف هاتفي') | Misclassified as positive | Not resolved; requires contextual models |
| Emoji Ambiguity (crying = laughing) | Emojis removed entirely | May lose genuine sentiment signal |
| Code-Switching (Arabic + English) | Latin characters removed | Reduces noise but loses some context |
| Imbalanced Classes | Model bias toward positive | class_weight=balanced + per-star sampling |
| Short Reviews | Low confidence predictions | min_df filter; acknowledged limitation |
| Temporal Bias | Older reviews may be stale | Sort.NEWEST used in collection |